In [ ]:
from IPython.display import HTML, display
display(HTML('''
<div style="text-align:right;padding:8px 0;">
<button id="toggle-code-btn" style="
  padding:8px 18px;font-size:14px;cursor:pointer;
  background:#4a90d9;color:white;border:none;border-radius:5px;
  box-shadow:0 2px 4px rgba(0,0,0,0.2);
" onclick="
  var cells = document.querySelectorAll('.jp-Cell-inputWrapper');
  var btn = document.getElementById('toggle-code-btn');
  var show = btn.dataset.show !== 'true';
  btn.dataset.show = show;
  btn.textContent = show ? '隐藏代码 [Hide]' : '显示代码 [Show]';
  cells.forEach(function(c){ c.style.display = show ? '' : 'none'; });
">显示代码 [Show]</button>
</div>
'''))


# 第8章 数据库（三）：SQL 语言
# Chapter 8 Databases (Part 3): SQL Language

---

**Cambridge International AS & A Level Computer Science (9618)**

> "SQL is the most successful programming language in history. It's the only language where you describe WHAT you want, not HOW to get it."

**SQL（Structured Query Language，结构化查询语言）** 是与关系数据库交互的标准语言。无论你使用 MySQL、PostgreSQL、SQLite 还是 Oracle，SQL 的核心语法都是相同的。

**本节学习目标：**
- 掌握 DDL（数据定义语言）：CREATE, ALTER, DROP
- 掌握 DML（数据操作语言）：INSERT, SELECT, UPDATE, DELETE
- 熟练使用 WHERE, ORDER BY, LIKE, GROUP BY 等子句
- 理解并使用聚合函数 COUNT, SUM, AVG, MIN, MAX
- 掌握 JOIN 操作（两表连接）
- 能够编写从简单到复杂的 SQL 查询

## 8.3.1 SQL 简史 | A Brief History of SQL

SQL 的历史与关系数据库紧密相连：

- **1970年：** E.F. Codd 发表论文提出关系模型
- **1974年：** IBM 开发了 SEQUEL（SQL 的前身）
- **1979年：** Oracle 发布第一个商业 SQL 数据库
- **1986年：** SQL 成为 ANSI/ISO 标准
- **今天：** SQL 仍是数据库领域的通用语言

### SQL 的两大分支

| 分支 | 全称 | 用途 | 关键词 |
|:---|:---|:---|:---|
| **DDL** | Data Definition Language | 定义数据库结构 | CREATE, ALTER, DROP |
| **DML** | Data Manipulation Language | 操作数据 | SELECT, INSERT, UPDATE, DELETE |

> **类比：**
> - DDL 就像**建筑设计**——决定房间的布局、大小、功能
> - DML 就像**日常生活**——在房间里放家具、移动物品、打扫卫生

## 8.3.2 准备工作：创建学习用数据库

我们先创建一个包含样本数据的数据库，后续所有 SQL 示例都基于它：

In [ ]:
import sqlite3

# We will use this connection throughout the notebook
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# Enable foreign key support in SQLite
cursor.execute('PRAGMA foreign_keys = ON')

print('SQLite database created in memory.')
print('Ready to learn SQL!')
print()
print('Note: In a real application, you would use:')
print("  conn = sqlite3.connect('school.db')  # saves to a file")
print('But for learning, in-memory is perfect.')

## 8.3.3 DDL — CREATE TABLE | 创建表

`CREATE TABLE` 是最基本的 DDL 语句，用于定义表的结构。

### 语法：
```sql
CREATE TABLE TableName (
    Column1 DataType Constraints,
    Column2 DataType Constraints,
    ...
    PRIMARY KEY (Column1),
    FOREIGN KEY (Column2) REFERENCES OtherTable(OtherColumn)
);
```

### 常用数据类型：

| SQL 数据类型 | 含义 | 示例 |
|:---|:---|:---|
| **INTEGER** | 整数 | 年龄、数量 |
| **REAL** / FLOAT | 浮点数 | 价格、分数 |
| **TEXT** | 文本字符串 | 姓名、地址 |
| **VARCHAR(n)** | 最大 n 个字符的字符串 | VARCHAR(50) |
| **BOOLEAN** | 布尔值 | TRUE / FALSE |
| **DATE** | 日期 | 2024-01-15 |

### 常用约束（Constraints）：

| 约束 | 作用 |
|:---|:---|
| **PRIMARY KEY** | 主键：唯一 + 非空 |
| **FOREIGN KEY** | 外键：引用另一表的主键 |
| **NOT NULL** | 不允许空值 |
| **UNIQUE** | 值必须唯一 |
| **CHECK** | 自定义条件检查 |
| **DEFAULT** | 默认值 |

In [ ]:
# DDL: CREATE TABLE examples

# Table 1: Departments
cursor.execute(
    'CREATE TABLE Departments ('
    '    DeptID TEXT PRIMARY KEY,'
    '    DeptName TEXT NOT NULL UNIQUE,'
    '    Location TEXT DEFAULT "Main Campus"'
    ')'
)
print('Created: Departments table')

# Table 2: Teachers
cursor.execute(
    'CREATE TABLE Teachers ('
    '    TeacherID TEXT PRIMARY KEY,'
    '    FirstName TEXT NOT NULL,'
    '    LastName TEXT NOT NULL,'
    '    DeptID TEXT,'
    '    Salary REAL CHECK(Salary > 0),'
    '    FOREIGN KEY (DeptID) REFERENCES Departments(DeptID)'
    ')'
)
print('Created: Teachers table')

# Table 3: Students
cursor.execute(
    'CREATE TABLE Students ('
    '    StudentID TEXT PRIMARY KEY,'
    '    FirstName TEXT NOT NULL,'
    '    LastName TEXT NOT NULL,'
    '    DateOfBirth TEXT,'
    '    Email TEXT UNIQUE,'
    '    YearGroup INTEGER CHECK(YearGroup BETWEEN 12 AND 13)'
    ')'
)
print('Created: Students table')

# Table 4: Courses
cursor.execute(
    'CREATE TABLE Courses ('
    '    CourseID TEXT PRIMARY KEY,'
    '    CourseName TEXT NOT NULL,'
    '    Credits INTEGER DEFAULT 3,'
    '    TeacherID TEXT,'
    '    FOREIGN KEY (TeacherID) REFERENCES Teachers(TeacherID)'
    ')'
)
print('Created: Courses table')

# Table 5: Enrollments (junction table)
cursor.execute(
    'CREATE TABLE Enrollments ('
    '    StudentID TEXT,'
    '    CourseID TEXT,'
    '    EnrollDate TEXT,'
    '    Score REAL,'
    '    Grade TEXT,'
    '    PRIMARY KEY (StudentID, CourseID),'
    '    FOREIGN KEY (StudentID) REFERENCES Students(StudentID),'
    '    FOREIGN KEY (CourseID) REFERENCES Courses(CourseID)'
    ')'
)
print('Created: Enrollments table')

print()
print('All 5 tables created successfully!')
print()
# Verify by listing all tables
cursor.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")
print('Tables in database:')
for table in cursor.fetchall():
    print(f'  {table[0]}')

### ALTER TABLE — 修改表结构

```sql
ALTER TABLE TableName ADD ColumnName DataType;
ALTER TABLE TableName RENAME TO NewName;
```

### DROP TABLE — 删除表

```sql
DROP TABLE TableName;
```

> **注意：** DROP TABLE 会永久删除表和所有数据！这是一个危险操作。

In [ ]:
# ALTER TABLE - Add a column
cursor.execute('ALTER TABLE Students ADD COLUMN PhoneNumber TEXT')
print('Added PhoneNumber column to Students table.')

# Verify the change
cursor.execute('PRAGMA table_info(Students)')
print()
print('Students table structure after ALTER:')
print(f"  {'Name':<15} {'Type':<10} {'NotNull':<8} {'PK':<4}")
print('  ' + '-' * 37)
for col in cursor.fetchall():
    print(f'  {col[1]:<15} {col[2]:<10} {col[3]:<8} {col[5]:<4}')

# DROP TABLE demo (create a temp table first)
print()
cursor.execute('CREATE TABLE TempTable (ID INTEGER PRIMARY KEY, Data TEXT)')
print('Created TempTable.')
cursor.execute('DROP TABLE TempTable')
print('Dropped TempTable.')
print()
print('DDL Summary:')
print('  CREATE TABLE - builds a new table')
print('  ALTER TABLE  - modifies an existing table')
print('  DROP TABLE   - permanently deletes a table')

## 8.3.4 DML — INSERT INTO | 插入数据

`INSERT INTO` 向表中添加新记录。

### 语法：
```sql
-- Insert with all columns (must match column order)
INSERT INTO TableName VALUES (value1, value2, ...);

-- Insert with specific columns (recommended - clearer!)
INSERT INTO TableName (Column1, Column2) VALUES (value1, value2);
```

In [ ]:
# DML: INSERT INTO

# Insert departments
cursor.executemany('INSERT INTO Departments (DeptID, DeptName, Location) VALUES (?, ?, ?)', [
    ('D01', 'Computer Science', 'Building A'),
    ('D02', 'Mathematics', 'Building A'),
    ('D03', 'English', 'Building B'),
    ('D04', 'Physics', 'Building C'),
])
print('Inserted 4 departments.')

# Insert teachers
cursor.executemany('INSERT INTO Teachers VALUES (?, ?, ?, ?, ?)', [
    ('T01', 'Wei', 'Wang', 'D01', 85000),
    ('T02', 'Jing', 'Li', 'D02', 82000),
    ('T03', 'Ming', 'Chen', 'D03', 78000),
    ('T04', 'Hua', 'Zhang', 'D01', 90000),
    ('T05', 'Lin', 'Liu', 'D04', 80000),
])
print('Inserted 5 teachers.')

# Insert students
cursor.executemany(
    'INSERT INTO Students (StudentID, FirstName, LastName, DateOfBirth, Email, YearGroup) '
    'VALUES (?, ?, ?, ?, ?, ?)', [
    ('S001', 'Alice', 'Johnson', '2007-03-15', 'alice@school.com', 12),
    ('S002', 'Bob', 'Smith', '2007-08-22', 'bob@school.com', 12),
    ('S003', 'Carol', 'Williams', '2006-12-01', 'carol@school.com', 13),
    ('S004', 'David', 'Brown', '2007-05-10', 'david@school.com', 12),
    ('S005', 'Eve', 'Davis', '2006-09-18', 'eve@school.com', 13),
    ('S006', 'Frank', 'Wilson', '2007-01-25', 'frank@school.com', 12),
    ('S007', 'Grace', 'Taylor', '2006-07-30', 'grace@school.com', 13),
    ('S008', 'Henry', 'Anderson', '2007-11-05', 'henry@school.com', 12),
])
print('Inserted 8 students.')

# Insert courses
cursor.executemany('INSERT INTO Courses VALUES (?, ?, ?, ?)', [
    ('CS101', 'Intro to Programming', 4, 'T01'),
    ('CS201', 'Data Structures', 4, 'T04'),
    ('MA101', 'Calculus', 4, 'T02'),
    ('EN101', 'Academic English', 3, 'T03'),
    ('PH101', 'Mechanics', 4, 'T05'),
])
print('Inserted 5 courses.')

# Insert enrollments
cursor.executemany('INSERT INTO Enrollments VALUES (?, ?, ?, ?, ?)', [
    ('S001', 'CS101', '2024-09-01', 92, 'A'),
    ('S001', 'MA101', '2024-09-01', 88, 'A-'),
    ('S001', 'EN101', '2024-09-01', 75, 'B'),
    ('S002', 'CS101', '2024-09-01', 85, 'B+'),
    ('S002', 'PH101', '2024-09-01', 90, 'A'),
    ('S003', 'CS201', '2024-09-01', 95, 'A+'),
    ('S003', 'MA101', '2024-09-01', 78, 'B+'),
    ('S004', 'CS101', '2024-09-01', 68, 'C+'),
    ('S004', 'EN101', '2024-09-01', 82, 'B'),
    ('S005', 'MA101', '2024-09-01', 98, 'A+'),
    ('S005', 'PH101', '2024-09-01', 91, 'A'),
    ('S006', 'CS101', '2024-09-01', 73, 'B-'),
    ('S006', 'CS201', '2024-09-01', 70, 'B-'),
    ('S007', 'EN101', '2024-09-01', 96, 'A+'),
    ('S007', 'MA101', '2024-09-01', 84, 'B+'),
    ('S008', 'CS101', '2024-09-01', 88, 'A-'),
    ('S008', 'PH101', '2024-09-01', 76, 'B'),
])
print('Inserted 17 enrollments.')

conn.commit()
print()
print('All sample data loaded! Ready to practice SQL queries.')

## 8.3.5 DML — SELECT | 查询数据

`SELECT` 是 SQL 中最常用、最强大的语句。从简单到复杂，让我们逐步学习：

### 基本语法：
```sql
SELECT Column1, Column2, ...
FROM TableName
WHERE condition
ORDER BY Column ASC|DESC;
```

### 特殊用法：
```sql
SELECT * FROM TableName;        -- * means ALL columns
SELECT DISTINCT Column FROM T;  -- Remove duplicates
```

In [ ]:
# SELECT: Basic queries

# Helper function to display query results nicely
def run_query(description, sql):
    print(f'--- {description} ---')
    print(f'SQL: {sql}')
    cursor.execute(sql)
    columns = [desc[0] for desc in cursor.description]
    rows = cursor.fetchall()
    # Print header
    header = '  '.join(f'{col:<15}' for col in columns)
    print(header)
    print('-' * len(header))
    for row in rows:
        print('  '.join(f'{str(val):<15}' for val in row))
    print(f'({len(rows)} rows)\n')

# Query 1: Select all columns from Students
run_query('All students', 'SELECT * FROM Students')

# Query 2: Select specific columns
run_query('Student names only', 'SELECT FirstName, LastName FROM Students')

# Query 3: Select with alias
run_query('With column alias',
    'SELECT FirstName AS "First Name", LastName AS "Last Name", YearGroup AS "Year" FROM Students')

### WHERE 子句 — 条件过滤

`WHERE` 让你只选择满足条件的记录。

| 运算符 | 含义 | 示例 |
|:---|:---|:---|
| `=` | 等于 | WHERE Age = 17 |
| `<>` 或 `!=` | 不等于 | WHERE Grade <> 'F' |
| `<`, `>`, `<=`, `>=` | 比较 | WHERE Score >= 90 |
| `AND` | 两个条件都为真 | WHERE Age > 16 AND Grade = 'A' |
| `OR` | 至少一个条件为真 | WHERE Grade = 'A' OR Grade = 'A+' |
| `NOT` | 取反 | WHERE NOT Grade = 'F' |
| `BETWEEN` | 在范围内（含两端） | WHERE Score BETWEEN 80 AND 90 |
| `IN` | 在列表中 | WHERE Grade IN ('A', 'A+', 'A-') |
| `IS NULL` | 为空 | WHERE Email IS NULL |

In [ ]:
# WHERE clause examples

# Query: Year 12 students
run_query('Year 12 students',
    'SELECT StudentID, FirstName, LastName, YearGroup FROM Students WHERE YearGroup = 12')

# Query: High scores (90+)
run_query('High scores (>= 90)',
    'SELECT StudentID, CourseID, Score, Grade FROM Enrollments WHERE Score >= 90')

# Query: AND - Year 12 students with names starting after 'D'
run_query('Year 12 students born after June 2007',
    "SELECT FirstName, LastName, DateOfBirth FROM Students WHERE YearGroup = 12 AND DateOfBirth > '2007-06-01'")

# Query: OR - Grade A+ or A
run_query('Grade A+ or A',
    "SELECT StudentID, CourseID, Score, Grade FROM Enrollments WHERE Grade = 'A+' OR Grade = 'A'")

# Query: BETWEEN
run_query('Scores between 80 and 90',
    'SELECT StudentID, CourseID, Score FROM Enrollments WHERE Score BETWEEN 80 AND 90')

# Query: IN
run_query('Grades in (A+, A, A-)',
    "SELECT StudentID, CourseID, Grade FROM Enrollments WHERE Grade IN ('A+', 'A', 'A-')")

### ORDER BY — 排序结果

```sql
SELECT ... FROM ... ORDER BY Column ASC;   -- Ascending (default)
SELECT ... FROM ... ORDER BY Column DESC;  -- Descending
SELECT ... FROM ... ORDER BY Col1 ASC, Col2 DESC;  -- Multiple columns
```

In [ ]:
# ORDER BY examples

run_query('Students ordered by LastName (A-Z)',
    'SELECT StudentID, FirstName, LastName FROM Students ORDER BY LastName ASC')

run_query('Enrollments ordered by Score (highest first)',
    'SELECT StudentID, CourseID, Score, Grade FROM Enrollments ORDER BY Score DESC')

run_query('Top 5 scores',
    'SELECT StudentID, CourseID, Score, Grade FROM Enrollments ORDER BY Score DESC LIMIT 5')

### LIKE — 模式匹配

`LIKE` 用于匹配文本模式，支持两个通配符：

| 通配符 | 含义 | 示例 |
|:---|:---|:---|
| `%` | 匹配任意数量的字符（0个或多个） | 'A%' 匹配 Alice, Anna, Aaron |
| `_` | 匹配恰好一个字符 | 'A__' 匹配 Amy, Ali（但不匹配 Alice） |

```sql
SELECT * FROM Students WHERE FirstName LIKE 'A%';    -- Starts with A
SELECT * FROM Students WHERE Email LIKE '%@school%';  -- Contains @school
SELECT * FROM Students WHERE FirstName LIKE '_o%';    -- Second letter is 'o'
```

In [ ]:
# LIKE pattern matching examples

run_query('Names starting with A or B',
    "SELECT FirstName, LastName FROM Students WHERE FirstName LIKE 'A%' OR FirstName LIKE 'B%'")

run_query('Names ending with e',
    "SELECT FirstName, LastName FROM Students WHERE FirstName LIKE '%e'")

run_query('5-letter first names (using _ wildcards)',
    "SELECT FirstName, LastName FROM Students WHERE FirstName LIKE '_____'")

run_query('Courses containing "to" or "ture"',
    "SELECT CourseID, CourseName FROM Courses WHERE CourseName LIKE '%t%'")

## 8.3.6 聚合函数 | Aggregate Functions

聚合函数对一组值进行计算，返回单个值。

| 函数 | 作用 | 示例 |
|:---|:---|:---|
| **COUNT(*)** | 计算行数 | SELECT COUNT(*) FROM Students |
| **COUNT(Column)** | 计算非空值的个数 | SELECT COUNT(Email) FROM Students |
| **SUM(Column)** | 求和 | SELECT SUM(Score) FROM Enrollments |
| **AVG(Column)** | 求平均值 | SELECT AVG(Score) FROM Enrollments |
| **MIN(Column)** | 最小值 | SELECT MIN(Score) FROM Enrollments |
| **MAX(Column)** | 最大值 | SELECT MAX(Score) FROM Enrollments |

In [ ]:
# Aggregate function examples

run_query('Total number of students',
    'SELECT COUNT(*) AS TotalStudents FROM Students')

run_query('Score statistics',
    'SELECT COUNT(*) AS NumEnrollments, '
    'AVG(Score) AS AvgScore, '
    'MIN(Score) AS MinScore, '
    'MAX(Score) AS MaxScore, '
    'SUM(Score) AS TotalScore '
    'FROM Enrollments')

run_query('Number of students per year group',
    'SELECT YearGroup, COUNT(*) AS NumStudents FROM Students GROUP BY YearGroup')

run_query('Average score per course',
    'SELECT CourseID, COUNT(*) AS Students, '
    'ROUND(AVG(Score), 1) AS AvgScore '
    'FROM Enrollments GROUP BY CourseID ORDER BY AvgScore DESC')

### GROUP BY 与 HAVING

- **GROUP BY** 将具有相同值的行分组，常与聚合函数配合使用
- **HAVING** 是对分组后的结果进行过滤（就像 WHERE 是对行过滤）

```sql
SELECT CourseID, AVG(Score)
FROM Enrollments
GROUP BY CourseID           -- 先按课程分组
HAVING AVG(Score) > 85;    -- 再筛选平均分 > 85 的课程
```

> **WHERE vs HAVING 的区别：**
> - `WHERE` 在分组**之前**过滤行
> - `HAVING` 在分组**之后**过滤组
> - `WHERE` 不能使用聚合函数，`HAVING` 可以

In [ ]:
# GROUP BY and HAVING examples

run_query('Average score per student (all students)',
    'SELECT StudentID, COUNT(*) AS Courses, '
    'ROUND(AVG(Score), 1) AS AvgScore '
    'FROM Enrollments GROUP BY StudentID ORDER BY AvgScore DESC')

run_query('Students with average score > 85',
    'SELECT StudentID, COUNT(*) AS Courses, '
    'ROUND(AVG(Score), 1) AS AvgScore '
    'FROM Enrollments GROUP BY StudentID HAVING AVG(Score) > 85 ORDER BY AvgScore DESC')

run_query('Courses with more than 3 students enrolled',
    'SELECT CourseID, COUNT(*) AS NumStudents '
    'FROM Enrollments GROUP BY CourseID HAVING COUNT(*) > 3')

print('Key difference:')
print('  WHERE filters ROWS before grouping')
print('  HAVING filters GROUPS after grouping')

## 8.3.7 DML — UPDATE | 更新数据

```sql
UPDATE TableName
SET Column1 = value1, Column2 = value2, ...
WHERE condition;
```

> **重要：** 永远记得加 `WHERE` 子句！没有 WHERE 的 UPDATE 会修改**所有行**！

In [ ]:
# UPDATE examples

# Show before update
run_query('Before UPDATE: Student S004',
    "SELECT * FROM Enrollments WHERE StudentID = 'S004' AND CourseID = 'CS101'")

# Update a student's score
cursor.execute("UPDATE Enrollments SET Score = 72, Grade = 'B-' WHERE StudentID = 'S004' AND CourseID = 'CS101'")
print(f'Rows updated: {cursor.rowcount}')
print()

# Show after update
run_query('After UPDATE: Student S004',
    "SELECT * FROM Enrollments WHERE StudentID = 'S004' AND CourseID = 'CS101'")

# Update multiple rows
cursor.execute("UPDATE Teachers SET Salary = Salary * 1.05 WHERE DeptID = 'D01'")
print(f'Gave 5% raise to CS department teachers. Rows updated: {cursor.rowcount}')
print()

run_query('CS department teachers after raise',
    "SELECT TeacherID, FirstName, LastName, Salary FROM Teachers WHERE DeptID = 'D01'")

## 8.3.8 DML — DELETE FROM | 删除数据

```sql
DELETE FROM TableName WHERE condition;
```

> **重要：** 没有 WHERE 的 DELETE 会删除**所有行**！这是极其危险的操作！

In [ ]:
# DELETE example

# First, let's add a temporary record to delete
cursor.execute("INSERT INTO Students (StudentID, FirstName, LastName, YearGroup) VALUES ('S099', 'Temp', 'Student', 12)")
print('Inserted temporary student S099.')

run_query('All students (including temp)',
    'SELECT StudentID, FirstName, LastName FROM Students ORDER BY StudentID')

# Delete the temporary student
cursor.execute("DELETE FROM Students WHERE StudentID = 'S099'")
print(f'Deleted temporary student. Rows deleted: {cursor.rowcount}')
print()

run_query('All students (temp removed)',
    'SELECT StudentID, FirstName, LastName FROM Students ORDER BY StudentID')

print('Remember: Always use WHERE with DELETE!')
print('  DELETE FROM Students;  -- THIS DELETES EVERYONE!!!')

## 8.3.9 JOIN — 连接查询 | Combining Tables

JOIN 是 SQL 最强大的功能之一——它让你可以把相关联的表**组合**在一起查询。

### 为什么需要 JOIN？

因为数据被规范化分散到了多个表中：
- 学生信息在 Students 表
- 课程信息在 Courses 表
- 选课关系在 Enrollments 表

当你想知道"每个学生选了什么课，得了多少分"时，就需要 JOIN 把这些表连起来。

### INNER JOIN 语法：

```sql
SELECT columns
FROM Table1
INNER JOIN Table2 ON Table1.key = Table2.key;
```

> **INNER JOIN** 只返回两个表中**都有匹配**的行。
>
> **类比：** 就像两份名单对照——只有两份名单上都出现的人才会被列出来。

In [ ]:
# JOIN examples - the most powerful SQL feature!

# JOIN 1: Students with their enrollment details
run_query('Students and their courses (INNER JOIN)',
    'SELECT s.FirstName, s.LastName, c.CourseName, e.Score, e.Grade '
    'FROM Students s '
    'INNER JOIN Enrollments e ON s.StudentID = e.StudentID '
    'INNER JOIN Courses c ON e.CourseID = c.CourseID '
    'ORDER BY s.LastName, c.CourseName')

# JOIN 2: Courses with their teacher names
run_query('Courses and their teachers',
    'SELECT c.CourseID, c.CourseName, t.FirstName, t.LastName, d.DeptName '
    'FROM Courses c '
    'INNER JOIN Teachers t ON c.TeacherID = t.TeacherID '
    'INNER JOIN Departments d ON t.DeptID = d.DeptID '
    'ORDER BY c.CourseID')

In [ ]:
# More JOIN examples with aggregation

# JOIN 3: Average score per course with course names
run_query('Average score per course (with names)',
    'SELECT c.CourseName, COUNT(*) AS Students, '
    'ROUND(AVG(e.Score), 1) AS AvgScore, '
    'MIN(e.Score) AS MinScore, MAX(e.Score) AS MaxScore '
    'FROM Enrollments e '
    'INNER JOIN Courses c ON e.CourseID = c.CourseID '
    'GROUP BY c.CourseName '
    'ORDER BY AvgScore DESC')

# JOIN 4: Each student's overall average
run_query('Each student overall average',
    'SELECT s.FirstName, s.LastName, s.YearGroup, '
    'COUNT(e.CourseID) AS NumCourses, '
    'ROUND(AVG(e.Score), 1) AS AvgScore '
    'FROM Students s '
    'INNER JOIN Enrollments e ON s.StudentID = e.StudentID '
    'GROUP BY s.StudentID '
    'ORDER BY AvgScore DESC')

# JOIN 5: Find students scoring above 90 in any course
run_query('Students with any score above 90',
    'SELECT DISTINCT s.FirstName, s.LastName, c.CourseName, e.Score '
    'FROM Students s '
    'INNER JOIN Enrollments e ON s.StudentID = e.StudentID '
    'INNER JOIN Courses c ON e.CourseID = c.CourseID '
    'WHERE e.Score > 90 '
    'ORDER BY e.Score DESC')

### JOIN 的工作原理（可视化）

```
STUDENTS table:                ENROLLMENTS table:
┌─────────┬──────────┐         ┌─────────┬────────┬───────┐
│StudentID│ FirstName│         │StudentID│CourseID│ Score │
├─────────┼──────────┤         ├─────────┼────────┼───────┤
│ S001    │ Alice    │────┐    │ S001   │ CS101  │  92   │
│ S002    │ Bob      │──┐ └───>│ S001   │ MA101  │  88   │
│ S003    │ Carol    │  └────->│ S002   │ CS101  │  85   │
└─────────┴──────────┘         └────────┴────────┴───────┘

INNER JOIN ON Students.StudentID = Enrollments.StudentID

Result:
┌──────────┬────────┬───────┐
│FirstName │CourseID│ Score │
├──────────┼────────┼───────┤
│ Alice    │ CS101  │  92   │  ← S001 matched
│ Alice    │ MA101  │  88   │  ← S001 matched
│ Bob      │ CS101  │  85   │  ← S002 matched
└──────────┴────────┴───────┘
```

每个 StudentID 在 Enrollments 中的每条匹配记录都会与 Students 表中对应的行组合。

## 8.3.10 从简单到复杂：查询构建步骤 | Building Complex Queries

当面对复杂查询需求时，按以下步骤构建：

1. **确定需要的列**（SELECT 什么）
2. **确定涉及的表**（FROM 哪些表）
3. **确定连接条件**（JOIN ON 什么）
4. **确定筛选条件**（WHERE 什么）
5. **确定分组方式**（GROUP BY 什么）
6. **确定分组过滤**（HAVING 什么）
7. **确定排序方式**（ORDER BY 什么）

### 完整 SQL 语句执行顺序：

```
FROM → JOIN → WHERE → GROUP BY → HAVING → SELECT → ORDER BY → LIMIT
```

> **注意：** SQL 的**书写顺序**和**执行顺序**是不同的！
> 书写：SELECT → FROM → JOIN → WHERE → GROUP BY → HAVING → ORDER BY
> 执行：FROM → JOIN → WHERE → GROUP BY → HAVING → SELECT → ORDER BY

In [ ]:
# Complex query building: step by step

print('=== Complex Query: Find Year 12 students with average score > 80 ===')
print()
print('Step 1: What columns do we need?')
print('  -> Student name, year group, number of courses, average score')
print()
print('Step 2: What tables are involved?')
print('  -> Students (for name, year), Enrollments (for scores)')
print()
print('Step 3: How do we join them?')
print('  -> ON Students.StudentID = Enrollments.StudentID')
print()
print('Step 4: What filter?')
print('  -> WHERE YearGroup = 12 (filter BEFORE grouping)')
print()
print('Step 5: How to group?')
print('  -> GROUP BY StudentID (one row per student)')
print()
print('Step 6: Group filter?')
print('  -> HAVING AVG(Score) > 80 (filter AFTER grouping)')
print()
print('Step 7: How to sort?')
print('  -> ORDER BY AvgScore DESC (highest first)')
print()

# The final query
run_query('Final result: Year 12 students with avg > 80',
    'SELECT s.FirstName, s.LastName, s.YearGroup, '
    'COUNT(e.CourseID) AS NumCourses, '
    'ROUND(AVG(e.Score), 1) AS AvgScore '
    'FROM Students s '
    'INNER JOIN Enrollments e ON s.StudentID = e.StudentID '
    'WHERE s.YearGroup = 12 '
    'GROUP BY s.StudentID '
    'HAVING AVG(e.Score) > 80 '
    'ORDER BY AvgScore DESC')

## 8.3.11 SQL 练习场 | SQL Playground

下面的代码单元提供了一个交互式 SQL 练习场。你可以修改 `your_query` 变量来尝试不同的查询：

In [ ]:
# SQL Playground - Change the query below and run!

# Available tables:
#   Departments(DeptID, DeptName, Location)
#   Teachers(TeacherID, FirstName, LastName, DeptID, Salary)
#   Students(StudentID, FirstName, LastName, DateOfBirth, Email, YearGroup, PhoneNumber)
#   Courses(CourseID, CourseName, Credits, TeacherID)
#   Enrollments(StudentID, CourseID, EnrollDate, Score, Grade)

# Try your own queries here!
your_query = (
    'SELECT s.FirstName, s.LastName, c.CourseName, e.Score '
    'FROM Students s '
    'INNER JOIN Enrollments e ON s.StudentID = e.StudentID '
    'INNER JOIN Courses c ON e.CourseID = c.CourseID '
    'WHERE e.Score = (SELECT MAX(Score) FROM Enrollments)'
)

try:
    cursor.execute(your_query)
    columns = [desc[0] for desc in cursor.description]
    rows = cursor.fetchall()
    header = '  '.join(f'{col:<15}' for col in columns)
    print(header)
    print('-' * len(header))
    for row in rows:
        print('  '.join(f'{str(val):<15}' for val in row))
    print(f'\n({len(rows)} rows returned)')
except Exception as e:
    print(f'Error: {e}')
    print('Check your SQL syntax and try again!')

## 8.3.12 SQL 命令速查表 | SQL Quick Reference

### DDL Commands:
```sql
CREATE TABLE name (col1 TYPE CONSTRAINT, ...);
ALTER TABLE name ADD col TYPE;
DROP TABLE name;
```

### DML Commands:
```sql
INSERT INTO table (col1, col2) VALUES (val1, val2);
SELECT col1, col2 FROM table WHERE condition ORDER BY col;
UPDATE table SET col1 = val1 WHERE condition;
DELETE FROM table WHERE condition;
```

### Useful Clauses:
```sql
WHERE col = value                     -- Exact match
WHERE col LIKE 'pattern%'            -- Pattern match
WHERE col BETWEEN val1 AND val2      -- Range
WHERE col IN (val1, val2, val3)      -- List
WHERE col IS NULL                    -- Null check
GROUP BY col                         -- Group rows
HAVING aggregate_condition           -- Filter groups
ORDER BY col ASC|DESC               -- Sort results
LIMIT n                             -- Limit rows
```

### Aggregate Functions:
```sql
COUNT(*), COUNT(col), SUM(col), AVG(col), MIN(col), MAX(col)
```

### JOIN:
```sql
SELECT ... FROM table1
INNER JOIN table2 ON table1.key = table2.key;
```

## 练习题 | Practice Exercises

使用上面创建的学校数据库，编写 SQL 查询来回答以下问题。在下面的代码单元中修改查询来练习！

### 练习 1（简单 SELECT）
找出所有 Year 13 的学生，显示他们的 FirstName 和 LastName。

### 练习 2（WHERE + ORDER BY）
找出所有分数低于 80 的选课记录，按分数从低到高排列。

### 练习 3（LIKE）
找出所有 CourseName 中包含 "to" 的课程。

### 练习 4（聚合 + GROUP BY）
计算每个学生选了多少门课，按课程数从多到少排列。

### 练习 5（JOIN）
显示每位教师的姓名及他们教的课程名称。

### 练习 6（JOIN + GROUP BY + HAVING）
找出平均成绩在 85 分以上的课程，显示课程名和平均成绩。

### 练习 7（综合挑战）
找出选修了 CS101 课程且分数排名前 3 的学生，显示他们的姓名和分数。

In [ ]:
# Practice exercise solutions

print('EXERCISE SOLUTIONS')
print('=' * 60)
print()

# Exercise 1
run_query('Exercise 1: Year 13 students',
    'SELECT FirstName, LastName FROM Students WHERE YearGroup = 13')

# Exercise 2
run_query('Exercise 2: Scores below 80',
    'SELECT StudentID, CourseID, Score, Grade FROM Enrollments WHERE Score < 80 ORDER BY Score ASC')

# Exercise 3
run_query('Exercise 3: Courses containing "to"',
    "SELECT CourseID, CourseName FROM Courses WHERE CourseName LIKE '%to%'")

# Exercise 4
run_query('Exercise 4: Courses per student',
    'SELECT StudentID, COUNT(*) AS NumCourses FROM Enrollments GROUP BY StudentID ORDER BY NumCourses DESC')

# Exercise 5
run_query('Exercise 5: Teachers and their courses',
    'SELECT t.FirstName, t.LastName, c.CourseName '
    'FROM Teachers t '
    'INNER JOIN Courses c ON t.TeacherID = c.TeacherID '
    'ORDER BY t.LastName')

# Exercise 6
run_query('Exercise 6: Courses with avg > 85',
    'SELECT c.CourseName, ROUND(AVG(e.Score), 1) AS AvgScore '
    'FROM Enrollments e '
    'INNER JOIN Courses c ON e.CourseID = c.CourseID '
    'GROUP BY c.CourseName '
    'HAVING AVG(e.Score) > 85 '
    'ORDER BY AvgScore DESC')

# Exercise 7
run_query('Exercise 7: Top 3 in CS101',
    'SELECT s.FirstName, s.LastName, e.Score '
    'FROM Students s '
    'INNER JOIN Enrollments e ON s.StudentID = e.StudentID '
    "WHERE e.CourseID = 'CS101' "
    'ORDER BY e.Score DESC LIMIT 3')

In [ ]:
# Clean up
conn.close()
print('Database connection closed.')
print()
print('Congratulations! You have learned:')
print('  DDL: CREATE TABLE, ALTER TABLE, DROP TABLE')
print('  DML: INSERT, SELECT, UPDATE, DELETE')
print('  Clauses: WHERE, ORDER BY, LIKE, GROUP BY, HAVING')
print('  Functions: COUNT, SUM, AVG, MIN, MAX')
print('  JOIN: INNER JOIN to combine tables')
print()
print('You are now ready to design and query real databases!')